In [ ]:
# pip install convokit

In [ ]:
SUBREDDITS_PER_LANGUAGE = {
    "de": ["de", "Finanzen"],
    "en": ["ukpolitics", "finance"],
    "es": ["es", "chile"],
    "fr": ["france"],
    "it": ["italy"],
    "pt": ["portugal", "brasil"],
    "uk": ["ukraina"],
    "ru": ["liberta"],
    "ar": ["jordan", "Egypt"],
    "bg": ["bulgaria"],
    "el": ["greece"],
    "pl": ["Polska"],
    "ca": ["catalunya"],
    "ja": ["newsokur"],
    "co": ["hanguk"]
}

In [ ]:
MIN_NUMBERS_OF_COMMENTS = 5
MAX_NESTED_SIZE = 5

from pydantic import BaseModel
from typing import List, Optional, Union
from tqdm import tqdm
from convokit import Corpus, download
from tqdm import tqdm
import joblib
import gc

class Message(BaseModel):
    id: Optional[str]
    message_text: str
    user_id: Union[int, str]
    answer_to_message_id: Optional[Union[int, str]]
    timestamp: Optional[int]
    metadata: Optional[dict]

    def __str__(self):
      return f"""Message ID={self.id}, USER_ID={self.user_id}, Answer to {self.answer_to_message_id} MESSAGE_TEXT={self.message_text}"""


class ThreadedConversation(BaseModel):
    id: Optional[str]
    initial_post: Optional[str]
    previous_comments: Optional[list[Message]]

    def __str__(self):
      line_dilimeter = "\n"
      return f"""ThreadedConversation ID={self.id}:
INITIAL POST: {self.initial_post}
PREVIOUS_COMMENTS:
{line_dilimeter.join([str(i) for i in self.previous_comments])}
"""


In [ ]:
for language in tqdm(SUBREDDITS_PER_LANGUAGE):
  for channel in SUBREDDITS_PER_LANGUAGE[language]:

    corpus = Corpus(download(f'subreddit-{channel}'))

    all_conversations = []
    number_of_processed_conversations = 0

    for conversation in tqdm(corpus.iter_conversations(), total=corpus.get_meta()['num_posts']):
      try:
        number_of_processed_conversations += 1
        if conversation.to_dict()["meta"]["num_comments"] > 5:  # --> Filter out active threads
          initial_post_id = conversation.get_id()
          initial_post_title = conversation.meta["title"]

          conversation.initialize_tree_structure()
          threaded_conversation_utts = conversation.tree.children
          # Iterate through threads in the post:
          for utt in threaded_conversation_utts:
            all_utts = utt.bfs_traversal()
            messages = []
            messages_to_pass = []
            # Iterate through all the messages in the thread:
            for utt in all_utts:
              # Check if message is not deleted (pass it and all messages that reply to it):
              if (utt.utt.text in ["[removed]", "[deleted]"]) or (utt.utt.reply_to in messages_to_pass):
                messages_to_pass.append(utt.utt.id)
                continue
              # Add the message to the conversation:
              metadata = {"score": utt.utt.meta['score']}
              messages.append(Message(id=utt.utt.id, message_text=utt.utt.text, user_id=utt.utt.speaker.id, answer_to_message_id=utt.utt.reply_to, timestamp=utt.utt.timestamp, metadata=metadata))

            if len(messages) > 0:
              threaded_conversation = ThreadedConversation(id=initial_post_id, initial_post=initial_post_title, previous_comments=messages)
              all_conversations.append(threaded_conversation)
      except:
          pass

    joblib.dump(all_conversations, f"../../data/reddit/01_raw_data/conversations-{language}-{channel}.joblib")
    corpus.print_summary_stats()

    del all_conversations
    del corpus
    gc.collect()